# ANN-2026 HW2

- Shouyue Liu

## Attempt to forecast the price of MSFT by analyzing the prices of multiple stocks, including MSFT, over several consecutive days leading up to the target day.
N.B. Different setup from HW1

In [18]:
from torch.utils.data import DataLoader, Dataset


class StockDataset(Dataset):

    def __init__(self, X, Y, days):
        self.X = X
        self.Y = Y.reshape(-1)
        self.days = days  # days ahead for prediction

    def __len__(self):
        return (len(self.Y) - self.days)

    def __getitem__(self, index):
        x = self.X[:, index:index + self.days]
        y = self.Y[index + self.days]
        return x, y


In [19]:
import numpy as np
from numpy import exp, sum, log, log10
import yfinance as yf
import pandas as pd


def get_price(tick, start='2020-01-01', end=None):
    return yf.Ticker(tick).history(start=start, end=end)['Close']


def get_prices(tickers, start='2020-01-01', end=None):
    df = pd.DataFrame()
    for s in tickers:
        df[s] = get_price(s, start, end)
    return df


feature_stocks = ['tsla', 'meta', 'nvda', 'amzn', 'nflx', 'gbtc', 'gdx', 'intc', 'dal', 'c', 'goog', 'aapl', 'msft', 'ibm', 'hp', 'orcl', 'sap', 'crm', 'hubs', 'twlo']
predict_stock = 'msft'

# getting data
start_date = '2020-01-01'

allX = get_prices(feature_stocks, start=start_date)
ally = get_prices([predict_stock], start=start_date)

In [20]:
import torch.utils.data as data
import torch

stockData = StockDataset(allX.to_numpy().transpose().astype(np.float32), ally.to_numpy().astype(np.float32), days=5)
train_set_size = int(len(stockData) * 0.7)  # 1c
valid_set_size = int(len(stockData) * 0.15)  # 1c
test_set_size = len(stockData) - train_set_size - valid_set_size

train_set, valid_set, test_set = data.random_split(stockData, [train_set_size, valid_set_size, test_set_size], generator=torch.Generator().manual_seed(42))

batch_size = train_set_size  # use entire dataset as batch
train_dataloader = DataLoader(train_set, batch_size=batch_size, shuffle=True)  # input:(20,5), label:1
valid_dataloader = DataLoader(valid_set, batch_size=batch_size, shuffle=False)
test_dataloader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

## 1. Build a simple MLP to forecast MSFT price using PyTorch Lightning.
You have total freedom of your MLP. But your MLP should take the last five day ($5 \times 20=100$) prices as input and you have to add dropout into your network.

### 1a. Create a subclass of pytorch_lightning.LightningModule. It should include \_\_init\_\_, training_step, validation_step, configure_optimizers in the class. (6 points)

In [21]:
import pytorch_lightning as pl
import torch
import torch.nn as nn


# simple MLP with dropout; input is 20 stocks × 5 days = 100 features
class MLP(pl.LightningModule):

    def __init__(self, input_dim=100, hidden_sizes=(128, 64), dropout=0.2, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(nn.Linear(input_dim, hidden_sizes[0]), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_sizes[0], hidden_sizes[1]), nn.ReLU(), nn.Dropout(dropout), nn.Linear(hidden_sizes[1], 1))
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        # x comes in as (batch, 20, days); flatten to (batch, 100)
        x = x.view(x.size(0), -1)
        return self.model(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log('train_mse', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log('val_mse', loss, prog_bar=True, on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log('test_mse', loss, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


### 1b. Create a subclass of pytorch_lightning.LightningDataModule. It should include \_\_init\_\_, train_dataloader, and val_dataloader in the class. (4 points)

In [22]:
import pytorch_lightning as pl

class StockDataModule(pl.LightningDataModule):

    def __init__(self, train_loader, val_loader, test_loader):
        super().__init__()
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.test_loader = test_loader

    def train_dataloader(self):
        return self.train_loader

    def val_dataloader(self):
        return self.val_loader

    def test_dataloader(self):
        return self.test_loader


### 1c. Complete the rest of the code and train the model with 70% of the data. You should set aside 15% of the data each for validation and testing.  Show the training and validation MSE (5 points)

In [23]:
# create datamodule and model, then fit
dm = StockDataModule(train_dataloader, valid_dataloader, test_dataloader)
model = MLP()

trainer = pl.Trainer(max_epochs=30, logger=False, enable_checkpointing=False)
trainer.fit(model, dm)

# show training and validation MSE after training
train_mse = trainer.callback_metrics['train_mse'].item()
val_results = trainer.validate(model, dataloaders=valid_dataloader, verbose=False)[0]
val_mse = val_results['val_mse']

print(f"Training MSE: {train_mse:.4f}")
print(f"Validation MSE: {val_mse:.4f}")

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name    | Type       | Params | Mode  | FLOPs
-------------------------------------------------------
0 | model   | Sequential | 21.2 K | train | 0    
1 | loss_fn | MSELoss    | 0      | train | 0    
-------------------------------------------------------
21.2 K    Trainable params
0         Non-trainable params
21.2 K    Total params
0.085     Total estimated model params size (MB)
9         Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/Users/shouyueliu/miniforge3/envs/ann_2026/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/Users/shouyueliu/miniforge3/envs/ann_2026/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
/Users/shouyueliu/miniforge3/envs/ann_2026/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=30` reached.


Validation: |          | 0/? [00:00<?, ?it/s]

Training MSE: 5838.7900
Validation MSE: 2556.7405


## 2. Construct a 1-D CNN to forecast MSFT stock price. You are free to use any design, but your network must consist of at least one convolutional layer and one dropout layer. You can also extend the duration leading up to the target day by modifying the "days" argument in the StockDataset. But "days" should not be larger than 32. (10 points)

In [24]:
# 2. 1-D CNN for MSFT forecasting
import torch
import torch.nn as nn
import pytorch_lightning as pl
import torch.utils.data as data
from torch.utils.data import DataLoader

cnn_days = 20  # must be <= 32

stockData_cnn = StockDataset(
    allX.to_numpy().transpose().astype(np.float32),
    ally.to_numpy().astype(np.float32),
    days=cnn_days,
)

train_set_size_cnn = int(len(stockData_cnn) * 0.7)
valid_set_size_cnn = int(len(stockData_cnn) * 0.15)
test_set_size_cnn = len(stockData_cnn) - train_set_size_cnn - valid_set_size_cnn

train_set_cnn, valid_set_cnn, test_set_cnn = data.random_split(
    stockData_cnn,
    [train_set_size_cnn, valid_set_size_cnn, test_set_size_cnn],
    generator=torch.Generator().manual_seed(42),
)

batch_size_cnn = min(64, train_set_size_cnn)
train_dataloader_cnn = DataLoader(train_set_cnn, batch_size=batch_size_cnn, shuffle=True)
valid_dataloader_cnn = DataLoader(valid_set_cnn, batch_size=batch_size_cnn, shuffle=False)
test_dataloader_cnn = DataLoader(test_set_cnn, batch_size=batch_size_cnn, shuffle=False)


class CNN1DRegressor(pl.LightningModule):

    def __init__(self, in_channels=20, dropout=0.3, lr=1e-3):
        super().__init__()
        self.save_hyperparameters()

        self.features = nn.Sequential(
            nn.Conv1d(in_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.AdaptiveAvgPool1d(1),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        x = self.features(x)
        return self.regressor(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log('train_mse', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log('val_mse', loss, prog_bar=True, on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log('test_mse', loss, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)


cnn_model = CNN1DRegressor(in_channels=len(feature_stocks), dropout=0.3, lr=1e-3)
trainer_cnn = pl.Trainer(max_epochs=40, logger=False, enable_checkpointing=False)
trainer_cnn.fit(cnn_model, train_dataloaders=train_dataloader_cnn, val_dataloaders=valid_dataloader_cnn)

train_mse_cnn = trainer_cnn.callback_metrics['train_mse'].item()
val_mse_cnn = trainer_cnn.validate(cnn_model, dataloaders=valid_dataloader_cnn, verbose=False)[0]['val_mse']
test_mse_cnn = trainer_cnn.test(cnn_model, dataloaders=test_dataloader_cnn, verbose=False)[0]['test_mse']

print(f"CNN days window: {cnn_days}")
print(f"CNN Training MSE: {train_mse_cnn:.4f}")
print(f"CNN Validation MSE: {val_mse_cnn:.4f}")
print(f"CNN Test MSE: {test_mse_cnn:.4f}")

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name      | Type       | Params | Mode  | FLOPs
---------------------------------------------------------
0 | features  | Sequential | 8.2 K  | train | 0    
1 | regressor | Sequential | 2.1 K  | train | 0    
2 | loss_fn   | MSELoss    | 0      | train | 0    
---------------------------------------------------------
10.3 K    Trainable params
0         Non-trainable params
10.3 K    Total params
0.041     Total estimated model params size (MB)
15        Modules in train mode
0         Modules in eval mode
0         Total Flops


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

`Trainer.fit` stopped: `max_epochs=40` reached.


Validation: |          | 0/? [00:00<?, ?it/s]

/Users/shouyueliu/miniforge3/envs/ann_2026/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

CNN days window: 20
CNN Training MSE: 3834.6826
CNN Validation MSE: 1958.1311
CNN Test MSE: 1868.2753


## 3. Please try to enhance the performance of the previously created MLP or CNN by applying hyperparameter tuning. You can use tools such as W&B hyperparameter sweep, SMAP, Optuna, or similar packages to achieve this. You need to optimize at least two parameters, with the dropout rate being one of them. (5 points)

In [25]:
# 3. Hyperparameter tuning (Optuna) on MLP
import torch
import torch.nn as nn
import pytorch_lightning as pl
from torch.utils.data import DataLoader
from pytorch_lightning.callbacks import EarlyStopping

import optuna


pl.seed_everything(42, workers=True)

# Use smaller mini-batches for more stable tuning.
tune_batch_size = min(64, len(train_set))
train_tune_dataloader = DataLoader(train_set, batch_size=tune_batch_size, shuffle=True)
valid_tune_dataloader = DataLoader(valid_set, batch_size=tune_batch_size, shuffle=False)
test_tune_dataloader = DataLoader(test_set, batch_size=tune_batch_size, shuffle=False)


class TunedMLP(pl.LightningModule):

    def __init__(self, input_dim=100, hidden_sizes=(128, 64), dropout=0.2, lr=1e-3, weight_decay=0.0):
        super().__init__()
        self.save_hyperparameters()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_sizes[0]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_sizes[0], hidden_sizes[1]),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_sizes[1], 1),
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, x):
        x = x.view(x.size(0), -1)
        return self.model(x).squeeze(1)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log('train_mse', loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log('val_mse', loss, prog_bar=True, on_step=False, on_epoch=True)

    def test_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.loss_fn(y_hat, y)
        self.log('test_mse', loss, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)


def objective(trial):
    hidden1 = trial.suggest_int('hidden1', 64, 256, step=64)
    hidden2 = trial.suggest_int('hidden2', 32, min(128, hidden1), step=32)
    dropout = trial.suggest_float('dropout', 0.1, 0.5)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    weight_decay = trial.suggest_float('weight_decay', 1e-8, 1e-3, log=True)

    model = TunedMLP(
        input_dim=len(feature_stocks) * 5,
        hidden_sizes=(hidden1, hidden2),
        dropout=dropout,
        lr=lr,
        weight_decay=weight_decay,
    )
    trainer = pl.Trainer(
        max_epochs=25,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        callbacks=[EarlyStopping(monitor='val_mse', mode='min', patience=5)],
    )
    trainer.fit(model, train_dataloaders=train_tune_dataloader, val_dataloaders=valid_tune_dataloader)
    val_mse_trial = trainer.validate(model, dataloaders=valid_tune_dataloader, verbose=False)[0]['val_mse']
    return float(val_mse_trial)


study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20, show_progress_bar=False)

best_params = study.best_params
print('Best hyperparameters:', best_params)
print(f"Best validation MSE from tuning: {study.best_value:.4f}")

best_model = TunedMLP(
    input_dim=len(feature_stocks) * 5,
    hidden_sizes=(best_params['hidden1'], best_params['hidden2']),
    dropout=best_params['dropout'],
    lr=best_params['lr'],
    weight_decay=best_params['weight_decay'],
)

trainer_best = pl.Trainer(
    max_epochs=40,
    logger=False,
    enable_checkpointing=False,
    callbacks=[EarlyStopping(monitor='val_mse', mode='min', patience=8)],
)
trainer_best.fit(best_model, train_dataloaders=train_tune_dataloader, val_dataloaders=valid_tune_dataloader)

best_train_mse = trainer_best.callback_metrics['train_mse'].item()
best_val_mse = trainer_best.validate(best_model, dataloaders=valid_tune_dataloader, verbose=False)[0]['val_mse']
best_test_mse = trainer_best.test(best_model, dataloaders=test_tune_dataloader, verbose=False)[0]['test_mse']

print(f"Tuned MLP Training MSE: {best_train_mse:.4f}")
print(f"Tuned MLP Validation MSE: {best_val_mse:.4f}")
print(f"Tuned MLP Test MSE: {best_test_mse:.4f}")

if 'val_mse' in globals():
    baseline_val = float(val_mse)
    print(f"Baseline MLP Validation MSE: {baseline_val:.4f}")
    print(f"Validation MSE improvement: {baseline_val - float(best_val_mse):.4f}")

Seed set to 42
[I 2026-03-04 17:38:04,114] A new study created in memory with name: no-name-c3a3c2a7-45e2-42aa-af6b-3d25c356dbce
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.

  | Name    | Type       | Params | Mode  | FLOPs
-------------------------------------------------------
0 | model   | Sequential | 17.1 K | train | 0    
1 | loss_fn | MSELoss    | 0      | train | 0    
-------------------------------------------------------
17.1 K    Trainable params
0         Non-trainable params
17.1 K    Total params
0.068     Total estimated model params size (MB)
9         Modules in train mode
0         Modules in eval mode
0         Total Flops
[I 2026-03-04 17:38:04,847] Trial 0 finished with value: 1845.955810546875 and param

Best hyperparameters: {'hidden1': 192, 'hidden2': 128, 'dropout': 0.1038659388414401, 'lr': 0.0006839431882255379, 'weight_decay': 9.451869498385703e-08}
Best validation MSE from tuning: 87.7915


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=40` reached.


Validation: |          | 0/? [00:00<?, ?it/s]

Testing: |          | 0/? [00:00<?, ?it/s]

Tuned MLP Training MSE: 567.7980
Tuned MLP Validation MSE: 75.1907
Tuned MLP Test MSE: 89.2530
Baseline MLP Validation MSE: 2556.7405
Validation MSE improvement: 2481.5498


### Model Design Challenge (Extra credit: Maximum 20 points) 🚀


Model & parameters
https://huggingface.co/iaaronlau/ANN2026

https://huggingface.co/iaaronlau/ANN2026-HW2-Challenge

Notebook:
https://github.com/iAaronLau/ANN2026/blob/master/hw2.challenge.ipynb